# Unicorn Time-to-Scale

**Question:** what predicts how fast a company reaches a one billion dollar valuation, and does the apparent acceleration over time survive scrutiny?

This notebook follows the build order in `unicorn-project-plan.md`. Cleaning decisions are documented in markdown cells as they are made, not retrofitted later.

## 1. First look

Before writing any cleaning logic, we look at the raw data as it actually is: shape, dtypes, and the first few rows. The project plan already tells us what to expect (negative durations, an old outlier, currency strings), but the point of this step is to verify that against the data itself rather than trust the summary.

In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

df = pd.read_csv('data/raw/unicorn_companies.csv')
df.shape

(1074, 10)

In [3]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 1074 entries, 0 to 1073
Data columns (total 10 columns):
 #   Column            Non-Null Count  Dtype
---  ------            --------------  -----
 0   Company           1074 non-null   str  
 1   Valuation         1074 non-null   str  
 2   Date Joined       1074 non-null   str  
 3   Industry          1074 non-null   str  
 4   City              1058 non-null   str  
 5   Country/Region    1074 non-null   str  
 6   Continent         1074 non-null   str  
 7   Year Founded      1074 non-null   int64
 8   Funding           1074 non-null   str  
 9   Select Investors  1073 non-null   str  
dtypes: int64(1), str(9)
memory usage: 84.0 KB


In [4]:
df.head()

,Company,Valuation,Date Joined,Industry,City,Country/Region,Continent,Year Founded,Funding,Select Investors
0,Bytedance,$180B,4/7/17,Artificial intelligence,Beijing,China,Asia,2012,$8B,"Sequoia Capital China, SIG Asia Investments, S..."
1,SpaceX,$100B,12/1/12,Other,Hawthorne,United States,North America,2002,$7B,"Founders Fund, Draper Fisher Jurvetson, Rothen..."
2,SHEIN,$100B,7/3/18,E-commerce & direct-to-consumer,Shenzhen,China,Asia,2008,$2B,"Tiger Global Management, Sequoia Capital China..."
3,Stripe,$95B,1/23/14,Fintech,San Francisco,United States,North America,2010,$2B,"Khosla Ventures, LowercaseCapital, capitalG"
4,Klarna,$46B,12/12/11,Fintech,Stockholm,Sweden,Europe,2005,$4B,"Institutional Venture Partners, Sequoia Capita..."


In [5]:
df.describe()

,Year Founded
count,1074.000000
mean,2012.895717
std,5.698573
min,1919.000000
25%,2011.000000
50%,2014.000000
75%,2016.000000
max,2021.000000


In [6]:
df.duplicated().sum()

np.int64(0)

In [7]:
df['Company'].duplicated().sum()

np.int64(1)

In [8]:
df[df['Company'].duplicated(keep=False)].sort_values('Company')

,Company,Valuation,Date Joined,Industry,City,Country/Region,Continent,Year Founded,Funding,Select Investors
40,Bolt,$11B,5/29/18,Auto & transportation,Tallinn,Estonia,Europe,2013,$1B,"Didi Chuxing, Diamler, TMT Investments"
44,Bolt,$11B,10/8/21,Fintech,San Francisco,United States,North America,2014,$1B,"Activant Capital, Tribe Capital, General Atlantic"


## Checking for other values

In [9]:
df['Valuation'].str[-1].unique()

<StringArray>
['B']
Length: 1, dtype: str

In [10]:
df['Funding'].str[-1].unique()

<StringArray>
['B', 'M', 'n']
Length: 3, dtype: str

In [11]:
df[df['Funding'].str[-1] == 'n']['Funding'].unique()

<StringArray>
['Unknown']
Length: 1, dtype: str

In [12]:
df['Funding'].isna().sum()

np.int64(0)

In [13]:
def parse_currency(value):
    if value == 'Unknown' or value == "unknown":
        return np.nan

    numeric_part = value.replace('$', '')

    if numeric_part.endswith('B'):
        return float(numeric_part[:-1]) * 1e9
    elif numeric_part.endswith('M'):
        return float(numeric_part[:-1]) * 1e6

    return np.nan

df['valuation_numeric'] = df['Valuation'].apply(parse_currency)
df['funding_numeric'] = df['Funding'].apply(parse_currency)

In [14]:
df[['Valuation', 'valuation_numeric']].head()

,Valuation,valuation_numeric
0,$180B,1.800000e+11
1,$100B,1.000000e+11
2,$100B,1.000000e+11
3,$95B,9.500000e+10
4,$46B,4.600000e+10


In [15]:
df[['Funding', 'funding_numeric']].sample(10)

,Funding,funding_numeric
1042,$257M,257000000.0
422,$129M,129000000.0
504,$525M,525000000.0
356,$224M,224000000.0
854,$300M,300000000.0
1064,$215M,215000000.0
534,$277M,277000000.0
123,$910M,910000000.0
693,$318M,318000000.0
947,Unknown,NaN


In [16]:
df['funding_numeric'].isna().sum()

np.int64(12)

In [17]:
(df['Funding'] == 'Unknown').sum()

np.int64(12)

In [18]:
df['date_joined_parsed'] = pd.to_datetime(df['Date Joined'], format='%m/%d/%y')

In [19]:
df['date_joined_parsed'].dt.year.min(), df['date_joined_parsed'].dt.year.max()

(np.int32(2007), np.int32(2022))

In [20]:
df['years_to_unicorn'] = df['date_joined_parsed'].dt.year - df['Year Founded']

In [21]:
df['years_to_unicorn'].describe()

count    1074.000000
mean        7.000931
std         5.329672
min        -4.000000
25%         4.000000
50%         6.000000
75%         9.000000
max        98.000000
Name: years_to_unicorn, dtype: float64

In [22]:
df[df['years_to_unicorn'] < 0][['Company', 'Date Joined', 'Year Founded', 'years_to_unicorn']]

,Company,Date Joined,Year Founded,years_to_unicorn
714,Yidian Zixun,10/17/17,2021,-4


In [23]:
df = df[df['years_to_unicorn'] >= 0].copy()
df.shape

(1073, 14)

In [24]:
df[df['Year Founded'] < 1990][['Company', 'Year Founded', 'Date Joined', 'years_to_unicorn']].sort_values('Year Founded')

,Company,Year Founded,Date Joined,years_to_unicorn
189,Otto Bock HealthCare,1919,6/24/17,98
373,Promasidor Holdings,1979,11/8/16,37
699,Five Star Business Finance,1984,3/26/21,37


In [25]:
df[df['Year Founded'] >= 1990]['Year Founded'].min()

np.int64(1990)

In [26]:
df = df[df['Year Founded'] >= 1990].copy()
df.shape

(1070, 14)

In [27]:
sorted(df['Country/Region'].unique())

['Argentina',
 'Australia',
 'Austria',
 'Bahamas',
 'Belgium',
 'Bermuda',
 'Brazil',
 'Canada',
 'Chile',
 'China',
 'Colombia',
 'Croatia',
 'Czech Republic',
 'Denmark',
 'Estonia',
 'Finland',
 'France',
 'Germany',
 'Hong Kong',
 'India',
 'Indonesia',
 'Ireland',
 'Israel',
 'Italy',
 'Japan',
 'Lithuania',
 'Luxembourg',
 'Malaysia',
 'Mexico',
 'Netherlands',
 'Nigeria',
 'Norway',
 'Philippines',
 'Senegal',
 'Singapore',
 'South Africa',
 'South Korea',
 'Spain',
 'Sweden',
 'Switzerland',
 'Thailand',
 'Turkey',
 'United Arab Emirates',
 'United Kingdom',
 'United States',
 'Vietnam']

In [28]:

df[df['Select Investors'].isna()][['Company', 'Industry', 'Select Investors']]

,Company,Industry,Select Investors
629,LinkSure Network,Mobile & telecommunications,NaN


In [29]:
def count_investors(value):
    if pd.isna(value):
        return np.nan
    return len(value.split(','))

df['investor_count'] = df['Select Investors'].apply(count_investors)

In [30]:
df[['Select Investors', 'investor_count']].sample(8)

,Select Investors,investor_count
79,"Sixth Street Partners, OrbiMed Advisors, Highl...",3.0
499,"Pantera Capital, QED Investors, Coinbase Ventures",3.0
664,"Jerusalem Venture Partners, Israel Growth Part...",3.0
662,"8VC, Menlo Ventures, Tiger Global Management",3.0
290,Goldman Sachs Asset Management,1.0
753,"Dila Capital, Framework Ventures, 3L",3.0
963,"Left Lane Capital, Galaxy Interactive, Tru Arr...",3.0
875,"QED Investors, DST Global, Endeavor",3.0


In [31]:
df['investor_count'].isna().sum()

np.int64(1)

In [32]:
top_tier_investors = [
    'Sequoia', 'Andreessen Horowitz', 'Tiger Global', 'Accel',
    'SoftBank', 'Founders Fund', 'Insight Partners'
]

def has_top_tier(value):
    if pd.isna(value):
        return np.nan
    value_lower = value.lower()
    return any(investor.lower() in value_lower for investor in top_tier_investors)

df['has_top_tier_investor'] = df['Select Investors'].apply(has_top_tier)

In [33]:
df['has_top_tier_investor'].value_counts(dropna=False)

has_top_tier_investor
False    708
True     361
NaN        1
Name: count, dtype: int64

In [34]:
cohort_bins = [1989, 1999, 2004, 2009, 2014, 2021]
cohort_labels = ['pre-2000', '2000-2004', '2005-2009', '2010-2014', '2015+']

df['founding_cohort'] = pd.cut(df['Year Founded'], bins=cohort_bins, labels=cohort_labels)

In [35]:
df['founding_cohort'].value_counts(dropna=False).sort_index()

founding_cohort
pre-2000      23
2000-2004     40
2005-2009    114
2010-2014    413
2015+        480
Name: count, dtype: int64

In [36]:
df['Industry'].value_counts()

Industry
Fintech                                224
Internet software & services           205
E-commerce & direct-to-consumer        111
Artificial intelligence                 73
Health                                  73
Other                                   57
Supply chain, logistics, & delivery     57
Cybersecurity                           50
Data management & analytics             41
Mobile & telecommunications             37
Hardware                                34
Auto & transportation                   31
Edtech                                  28
Consumer & retail                       24
Travel                                  14
Artificial Intelligence                 11
Name: count, dtype: int64

In [37]:
df['Industry'] = df['Industry'].str.lower().str.capitalize()

In [38]:
df['Industry'].value_counts()

Industry
Fintech                                224
Internet software & services           205
E-commerce & direct-to-consumer        111
Artificial intelligence                 84
Health                                  73
Other                                   57
Supply chain, logistics, & delivery     57
Cybersecurity                           50
Data management & analytics             41
Mobile & telecommunications             37
Hardware                                34
Auto & transportation                   31
Edtech                                  28
Consumer & retail                       24
Travel                                  14
Name: count, dtype: int64

In [39]:
threshold = len(df) * 0.03
industry_counts = df['Industry'].value_counts()
small_industries = industry_counts[industry_counts < threshold].index.tolist()
small_industries

['Auto & transportation', 'Edtech', 'Consumer & retail', 'Travel']

In [40]:
df['industry_grouped'] = df['Industry'].apply(
    lambda x: 'Other' if x in small_industries else x
)

df['industry_grouped'].value_counts()

industry_grouped
Fintech                                224
Internet software & services           205
Other                                  154
E-commerce & direct-to-consumer        111
Artificial intelligence                 84
Health                                  73
Supply chain, logistics, & delivery     57
Cybersecurity                           50
Data management & analytics             41
Mobile & telecommunications             37
Hardware                                34
Name: count, dtype: int64

In [41]:
df['Continent'].value_counts(dropna=False)

Continent
North America    589
Asia             307
Europe           142
South America     21
Oceania            8
Africa             3
Name: count, dtype: int64

In [42]:
continent_threshold = len(df) * 0.03
continent_counts = df['Continent'].value_counts()
small_continents = continent_counts[continent_counts < continent_threshold].index.tolist()
small_continents

['South America', 'Oceania', 'Africa']

In [43]:
df['continent_grouped'] = df['Continent'].apply(
    lambda x: 'Other' if x in small_continents else x
)

df['continent_grouped'].value_counts()

continent_grouped
North America    589
Asia             307
Europe           142
Other             32
Name: count, dtype: int64